In [1]:
# Goal: Generate timeseries data from realistic load profiles (df_load_buses.parquet)

# Steps:
# 1. Pick n load buses randomly from the realistic load profiles (df_load_buses.parquet)
    # - load_buses.parquet load profiles for 2000 buses. 
    # - The amount of load buses needed is derived from the selected IEEE case (e.g. IEEE case 14 ⇒  11 load buses) 
# 2. Normalise every load bus profile individually 
    # - normalise every bus load profile by dividing every load by its buses max_load
# 3. Mulitply the normalised bus load profiles with the Active Power (P) and Reactive Power (Q), derived from the selected IEEE case
    # - for case 14: The non-load buses Bus 0, bus 6 and bus 7 (in zero indexing) will have P and Q zero
# 4. Formatting: Use the normalised load buses timesieries to create the precomputed_profile.csv
    # - Formatting defined in load_pertubation.py → *class* *PrecomputedProfile(LoadScenarioGeneratorBase)*
# 5 Generate Output via gridfm-datakit with config.yaml:
    # load: generator: precomputed_profile
    # Added argument: scenario_file: "/path/to/load-scenarios/load-scenarios-precomputed-temp.csv" # precomputed scenarios (cols: load_scenario, load, p_mw, q_mvar)
    # All pertubations set to None

In [2]:
# --- 1. Configuration ---
CASE_NAME = "case118"  # e.g., "case14", "case30", "case118"
LOAD_PROFILE_PATH = "D:/Data/studium/Master/MA_Code/data/updated_load_profiles/df_load_bus_2019-2021.parquet"
OUTPUT_FILE = f"{CASE_NAME}_ieee_3yr_precomputed_load_profiles.csv"
RANDOM_SEED = 42

# --- 2. Imports ---
import pandas as pd
import numpy as np
from matpowercaseframes import CaseFrames

# --- 3. Helper Functions ---
def get_ieee_base_loads(case_name):
    """Returns a DataFrame with base P (MW) and Q (MVar) for all buses of the given IEEE/Matpower test case."""
    cf = CaseFrames(case_name)
    bus = cf.bus
    base_df = pd.DataFrame({
        'bus_idx': np.arange(len(bus)),
        'P_base':  bus['PD'].values,
        'Q_base':  bus['QD'].values,
    })
    return base_df

# --- 4. Load and Prepare Data ---
ieee_base = get_ieee_base_loads(CASE_NAME)
df = pd.read_parquet(LOAD_PROFILE_PATH)
N_BUSES = (ieee_base['P_base'] != 0).sum()  # count of actual load buses

# sample N_BUSES
wide_df = df.pivot(index='timestamp', columns='bus', values='load_corrected').sort_index()
sampled_df = wide_df.sample(n=N_BUSES, axis=1, random_state=RANDOM_SEED)

# normalise
norm_profiles = (sampled_df / sampled_df.max()).values
n_scenarios = len(norm_profiles)

print(f"Time steps: {n_scenarios}, \n normalised Shape: {norm_profiles.shape}")

Time steps: 26280, 
 normalised Shape: (26280, 99)


In [3]:
# Preview sampled and normalized data
sampled_df.head()

bus,8102,8122,6217,3075,7409,5344,6355,5274,8145,7261,...,7071,5209,4144,5176,5084,5085,2121,8014,1018,5125
timestamp,,,,,,,,,,,,,,,,,,,,,
2019-01-01 00:30:00+00:00,6.558355,7.852584,19.753671,2.915160,118.758742,18.157400,46.053502,2.394593,36.046741,4.618631,...,30.068521,41.559546,12.329093,115.897960,61.188112,46.616158,0.473498,3.439178,130.772394,115.621227
2019-01-01 01:30:00+00:00,6.686820,8.255458,20.907795,3.234379,122.920149,18.671080,48.872725,2.515248,37.079099,4.696488,...,30.927718,42.733523,12.875931,120.698962,60.987947,48.107617,0.510950,3.553145,133.074781,117.202864
2019-01-01 02:30:00+00:00,6.779658,8.549566,21.947136,3.488743,126.281045,19.137122,51.280192,2.617698,37.685425,4.721128,...,31.492429,43.417090,13.245772,123.468119,60.975266,48.965155,0.548171,3.625077,134.511718,118.522469
2019-01-01 03:30:00+00:00,6.751110,8.484631,22.074657,3.514999,125.379425,19.121373,51.400199,2.632800,37.062322,4.647775,...,30.676595,43.232390,12.891279,122.773945,60.688689,48.699886,0.533598,3.556655,134.832111,117.905702
2019-01-01 04:30:00+00:00,6.582503,8.064063,21.317146,3.334722,119.425422,18.410414,48.986764,2.553792,35.190776,4.467881,...,28.334438,41.714468,11.868021,118.230633,58.735509,46.962769,0.490665,3.370279,131.027699,113.451749


In [4]:
# --- 5. Generate Output ---
# Multiply realistic load profiles with IEEE case values (P,Q) -> create output file
ieee = ieee_base.sort_values('bus_idx')
n_buses_case = len(ieee)
load_mask = (ieee['P_base'] != 0).values  # Boolean mask for the load buses

P_mat = np.zeros((n_scenarios, n_buses_case))
Q_mat = np.zeros((n_scenarios, n_buses_case))
P_mat[:, load_mask] = norm_profiles * ieee.loc[load_mask, 'P_base'].values
Q_mat[:, load_mask] = norm_profiles * ieee.loc[load_mask, 'Q_base'].values

out_df = pd.DataFrame({
    'load_scenario': np.repeat(np.arange(n_scenarios), n_buses_case),
    'load':          np.tile(ieee['bus_idx'].values, n_scenarios),
    'p_mw':          P_mat.flatten(),  # Flatten -> (Time 0 [Bus0..n], Time 1 [Bus0..n]...)
    'q_mvar':        Q_mat.flatten()
})

out_df.to_csv(OUTPUT_FILE, index=False)
print(f"Output_df {len(out_df)} rows.")
print(f"Expected rows: {n_scenarios * n_buses_case}")

Output_df 3101040 rows.
Expected rows: 3101040


In [5]:

out_df.head()

,load_scenario,load,p_mw,q_mvar
0,0,0,28.348504,15.008031
1,0,1,8.887958,3.999581
2,0,2,16.905868,4.334838
3,0,3,14.808911,4.556588
4,0,4,0.000000,0.000000
